# Create the Final Dataset with Weather + Station + POI + Trip Data

### Process
1. Load all CSVs
2. Remove trips with invalid station IDs
3. Extract temporal features from trip timestamps ← Do this here
4. Create complete time skeleton (all hours × all active stations)
5. Aggregate trips by [date, hour, station_id] → rides_started, rides_ended
6. Join to time skeleton (fill nulls with 0) ← Include zero-demand hours
7. Join weather data by [date, hour]
8. Join station metadata by station_id
9. Add cyclical encodings for temporal features
10. Validate data quality
11. Export CSV



### Load CSVs and Initial Inspection

In [ ]:
import pandas as pd

bike_trips_df = pd.read_csv('all_bike_trips.csv')
weather_df = pdf.read_csv('all_weather.csv')
stations_df = pd.read_csv('final_stations.csv')

print("=== Bike Trips Data ===")
print(bike_trips_df.head())
print(f"\nShape: {bike_trips_df.shape}")
print(f"Columns: {bike_trips_df.columns.tolist()}")

print("\n=== Weather Data ===")
print(weather_df.head())
print(f"\nShape: {weather_df.shape}")
print(f"Columns: {weather_df.columns.tolist()}")

print("\n=== Stations Data ===")
print(stations_df.head())
print(f"\nShape: {stations_df.shape}")
print(f"Columns: {stations_df.columns.tolist()}")


### Convert Date Columns to Datetime

# Convert datetime columns for trips
bike_trips_df['Start Time'] = pd.to_datetime(bike_trips_df['Start Time'])
bike_trips_df['End Time'] = pd.to_datetime(bike_trips_df['End Time'])

# Convert weather datetime
weather_df['Date/Time'] = pd.to_datetime(weather_df['Date/Time'])

print("Date columns converted to datetime")
print(f"Trips date range: {bike_trips_df['Start Time'].min()} to {bike_trips_df['Start Time'].max()}")
print(f"Weather date range: {weather_df['Date/Time'].min()} to {weather_df['Date/Time'].max()}")


### Remove Trips with Invalid Stations

# Get valid station IDs
valid_station_ids = set(stations_df['station_id'].unique())

print(f"Total valid stations: {len(valid_station_ids)}")

# Count trips before filtering
initial_trip_count = len(bike_trips_df)

# Filter trips to only include valid stations
bike_trips_df = bike_trips_df[
    bike_trips_df['Start Station Id'].isin(valid_station_ids) & 
    bike_trips_df['End Station Id'].isin(valid_station_ids)
]

trips_removed = initial_trip_count - len(bike_trips_df)
print(f"\nTrips before filtering: {initial_trip_count:,}")
print(f"Trips after filtering: {len(bike_trips_df):,}")
print(f"Trips removed: {trips_removed:,} ({trips_removed/initial_trip_count*100:.2f}%)")